In [ ]:
import base64
from IPython.display import Image, display
import matplotlib.pyplot as plt


def mm(graph):
  graphbytes = graph.encode("ascii")
  base64_bytes = base64.b64encode(graphbytes)
  base64_string = base64_bytes.decode("ascii")
  display(Image(url="https://mermaid.ink/svg/" + base64_string))

In [ ]:
mm("""
   graph TD
A[fa:fa-database JetBrains Datastore 10M Dataset] 
    -->|Download ntbs_list.json, pick 100K notebooks|B
B[fa:fa-database 100k-dataset.csv]
    -->|fetch_notebooks.py| C[fa:fa-book Download 100K Notebooks]
C -->|Store| D[fa:fa-file Data 100K Notebooks]
D -->|export_notebooks_to_html.py| E{HTML files by Theme}
E --> F[Darcula]
E --> G[Horizon]
E --> H[Material Darker]
E --> I[Solarized]
E --> J[Light]
E --> K[Dark]
J --> 
    |process_notebook_html.py|L(nb_processed_cell_html.csv)
L -->
    |get_first_interactive_cells.py|M(nb_first_interactive_cell.csv)
F -->N{Scan}
G -->N
H -->N
I -->N
J -->N
K -->N
N -->|pa11y_scan_parallel.py|X[pa11y-results]
X -->|fetch_a11y_responses.py|O{Results}
O -->P(a11y-aggregate-scan.csv)
O -->Q(a11y-detailed-result.csv)
Q -->|generate_accessibility_error_counts.py|R(errors-different-counts-a11y-analyze-errors-summary.csv)
D -->|process_notebooks.py|S[Base64 Images]
S -->|classify_images.py|T(model-results.csv)
D -->|process_notebooks.py|U(nb_processed.csv)
U -->|analyze_function_calls.py|V(processed_function_calls.csv)
   """
   )

In [ ]:
import pandas as pd

# Load the CSV file into a DataFrame
df = pd.read_csv('/Users/ainsleymartinez/Downloads/a11y-aggregate-scan.csv')


In [ ]:
# Ensure numerical columns are treated as numbers
df['Total'] = pd.to_numeric(df['Total'], errors='coerce')
df['Errors'] = pd.to_numeric(df['Errors'], errors='coerce')
df['Warnings'] = pd.to_numeric(df['Warnings'], errors='coerce')

# Calculate the proportions
df['Error Proportion'] = df['Errors'] / df['Total']
df['Warning Proportion'] = df['Warnings'] / df['Total']


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Select relevant columns for the heatmap
heatmap_data = df[['Error Proportion', 'Warning Proportion']]

# Create a correlation matrix for the heatmap
correlation_matrix = heatmap_data.corr()

# Plot the heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Heatmap of Error and Warning Proportions')
plt.show()


In [ ]:
df

In [ ]:
import pandas as pd

# Load a sample of the data to check the structure
df = pd.read_csv('/Users/ainsleymartinez/Downloads/a11y-aggregate-scan.csv', nrows=1000)


In [ ]:
print(df.head())
print(df.columns)


In [ ]:
# Ensure numeric columns are in the correct format
df['Errors'] = pd.to_numeric(df['Errors'], errors='coerce')
df['Warnings'] = pd.to_numeric(df['Warnings'], errors='coerce')
df['Total'] = pd.to_numeric(df['Total'], errors='coerce')

# Drop rows with missing values in these columns
df = df.dropna(subset=['Errors', 'Warnings', 'Total'])


In [ ]:
aggregated_df = df.groupby('Notebook').agg({
    'Errors': 'sum',
    'Warnings': 'sum',
    'Total': 'sum'
}).reset_index()


In [ ]:
import matplotlib.pyplot as plt

# Create a bar plot comparing Errors and Warnings to Total
plt.figure(figsize=(12, 8))

# Plot Errors and Warnings as a fraction of Total
plt.bar(aggregated_df['Notebook'], aggregated_df['Errors'] / aggregated_df['Total'], label='Errors')
plt.bar(aggregated_df['Notebook'], aggregated_df['Warnings'] / aggregated_df['Total'], bottom=aggregated_df['Errors'] / aggregated_df['Total'], label='Warnings')

plt.xlabel('Notebook')
plt.ylabel('Proportion of Total')
plt.title('Proportion of Errors and Warnings to Total Issues')
plt.xticks(rotation=90)
plt.legend()
plt.tight_layout()

plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the data
df = pd.read_csv('/Users/ainsleymartinez/Downloads/a11y-aggregate-scan.csv', nrows=1000)

# Convert relevant columns to numeric and handle missing values
df['Errors'] = pd.to_numeric(df['Errors'], errors='coerce')
df['Warnings'] = pd.to_numeric(df['Warnings'], errors='coerce')
df['Total'] = pd.to_numeric(df['Total'], errors='coerce')
df = df.dropna(subset=['Errors', 'Warnings', 'Total'])

# Classify each entry
df['Category'] = 'Successful'
df.loc[df['Errors'] > 0, 'Category'] = 'Errors'
df.loc[df['Warnings'] > 0, 'Category'] = 'Warnings'

# Prepare data for plotting
df['x'] = df.index  # Using index for x-axis positioning
df['y'] = 0  # Constant y value for dot plot

# Define colors for categories
colors = {'Errors': 'red', 'Warnings': 'orange', 'Successful': 'blue'}

# Create the plot
plt.figure(figsize=(12, 8))

# Plot dots for each category
for category, color in colors.items():
    subset = df[df['Category'] == category]
    plt.scatter(subset['x'], subset['y'], color=color, label=category, alpha=0.6, edgecolor='k', s=100)

# Add labels and title
plt.yticks([])  # Remove y-axis ticks as it's a dot plot
plt.xlabel('Notebook Index')
plt.title('Dot Plot of Errors, Warnings, and Successful Notebooks')
plt.legend(title='Category')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()

# Show the plot
plt.show()


In [ ]:
df['Total Issues'] = df['Errors'] + df['Warnings'] + df['Notices']

# Calculate the percentage of errors
df['Error Percentage'] = (df['Errors'] / df['Total Issues']) * 100

# Print the result
print(df[['ID', 'Error Percentage']])

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy
import pandas
import seaborn as sns

matplotlib.rcParams.update({'font.size': 20, 'pdf.fonttype': 42, 'ps.fonttype': 42})

In [ ]:
different_errors = pandas.read_csv('/Users/ainsleymartinez/Downloads/errors-different-counts-a11y-analyze-errors-summary.csv')

In [ ]:
error_grid = different_errors[different_errors['Type'] == 'error'].pivot(index='Theme', columns=['Runner', 'DetailCode'], values='count')


In [ ]:
axe_errors = error_grid['axe']
axe_error_columns = axe_errors.columns

reassignment = {}

for i, column_name in enumerate(axe_error_columns):
    reassignment[column_name] = f'AXE-E{i+1}'

reassignment_df = pandas.DataFrame.from_dict(reassignment, orient="index")
reassignment_df.rename(columns={0: 'Error Code'}, inplace=True)
reassignment_df

In [ ]:
axe_errors.rename(columns=reassignment, inplace=True)
axe_errors

In [ ]:
for updated_column in axe_errors.columns.tolist():
    normalizer = axe_errors[updated_column].max()
    axe_errors[updated_column] = axe_errors[updated_column] / normalizer * 100.0
    
axe_errors

In [ ]:
fig7a_description = """Figure 7(a) The heatmap of errors for aXe indicates the percentage of errors of different 
types. The type of errors are presented on the x axis and the themes along the y axis with each cell ranging 
between 0 and 100.0. The corresponding legend of the color bar is to the right side of the figure.
"""

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10, 8))
sns.heatmap(axe_errors, robust=True, linewidths=0.1, linecolor='black', ax=ax, vmin=0, vmax=100)
ax.set_ylabel('Notebook Theme', fontsize=20)
ax.set_xlabel('Errors Reported by Axe Accessibility Engine', fontsize=20)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.savefig('plot_out/fig-7a-errors-axe-heatmap.pdf', bbox_inches='tight')
plt.savefig('plot_out/alt-embedded-images/fig-7a-errors-axe-heatmap.png',
            metadata={'alt': fig7a_description},
            bbox_inches='tight')